# ARC-AGI-3 Qwen portfolio mini-wave

Private non-scored development only. 6 games with `ARC3_PORTFOLIO_SECONDS=5400` and `ARC3_TRACE=/kaggle/working/coverage.jsonl`.
`concurrency=1` for true sequential portfolio notify→next-apply.
TRUE_SUBMISSION must stay false — never a scored competition submit.


In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1"

# Gate 1 safe baseline: the prior C8 run is the comparison; validate C16 here.
PUBLIC25_VLLM_PROFILE_NAME = 'kv5-bf16-mtp3-c16-cg32'
PUBLIC25_VLLM_PROFILE_ENV = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "16",
    "TAAF_VLLM_MTP_TOKENS": "3",
    "TAAF_VLLM_OMP_THREADS": "1"
}
for key, value in PUBLIC25_VLLM_PROFILE_ENV.items():
    os.environ[key] = value
print(
    f'PUBLIC25_VLLM_PROFILE name={PUBLIC25_VLLM_PROFILE_NAME} '
    f'env={json.dumps(PUBLIC25_VLLM_PROFILE_ENV, sort_keys=True)}',
    flush=True,
)
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")
if TRUE_SUBMISSION:
    raise RuntimeError("Qwen portfolio smoke must never run as a competition submission.")
os.environ['ARC3_TRACE'] = str(WORKING_DIR / 'coverage.jsonl')
os.environ['ARC3_PORTFOLIO_SECONDS'] = '5400'
os.environ.setdefault('ARC3_SCORED_ENVIRONMENTS', '25')
os.environ.setdefault('ARC3_MIN_SECONDS_PER_GAME', '90')
print(
    f"PORTFOLIO_MINIWAVE_ENV trace={os.environ['ARC3_TRACE']} "
    f"portfolio_seconds={os.environ['ARC3_PORTFOLIO_SECONDS']}",
    flush=True,
)


In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Embed arc3 portfolio + JSONL trace helpers (offline Kaggle).
import sys as _arc3_sys
from pathlib import Path as _arc3_Path
_ARC3_PKG = WORKING_DIR / 'arc3'
_ARC3_PKG.mkdir(parents=True, exist_ok=True)
(_ARC3_PKG / '__init__.py').write_text('', encoding='utf-8')
(_ARC3_PKG / 'instrumentation.py').write_text('"""Per-game run instrumentation for coverage and depth decisions.\n\nEvery competed environment should emit one record so we can answer:\nhow many games were touched, how deep, why abandoned, and where wall time went.\nUntouched scored environments still contribute 0 to the mean — coverage gaps\nare first-class signal, not an afterthought.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\n\nSCHEMA_VERSION = 1\n\n\n@dataclass\nclass GameTrace:\n    game_id: str\n    wall_seconds: float = 0.0\n    actions: int = 0\n    levels_completed: int = 0\n    level_indices_completed: list[int] = field(default_factory=list)\n    abandon_reason: str = ""\n    tokens_in: int = 0\n    tokens_out: int = 0\n    llm_calls: int = 0\n    score: float | None = None\n    meta: dict[str, Any] = field(default_factory=dict)\n\n    def to_dict(self) -> dict[str, Any]:\n        payload = asdict(self)\n        payload["schema_version"] = SCHEMA_VERSION\n        return payload\n\n\nclass TraceRecorder:\n    """Append-only JSONL + optional final JSON summary."""\n\n    def __init__(self, path: str | Path) -> None:\n        self.path = Path(path)\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        if self.path.exists():\n            self.path.unlink()\n        self._rows: list[dict[str, Any]] = []\n\n    def record(self, trace: GameTrace) -> None:\n        row = trace.to_dict()\n        self._rows.append(row)\n        with self.path.open("a", encoding="utf-8") as handle:\n            handle.write(json.dumps(row, sort_keys=True) + "\\n")\n\n    def coverage_summary(self, scored_environments: int) -> dict[str, Any]:\n        touched = {r["game_id"] for r in self._rows}\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "scored_environments": scored_environments,\n            "games_touched": len(touched),\n            "games_untouched": max(0, scored_environments - len(touched)),\n            "games_with_zero_levels": sum(\n                1 for r in self._rows if int(r.get("levels_completed") or 0) == 0\n            ),\n            "total_levels_completed": sum(\n                int(r.get("levels_completed") or 0) for r in self._rows\n            ),\n            "total_wall_seconds": sum(\n                float(r.get("wall_seconds") or 0) for r in self._rows\n            ),\n            "total_actions": sum(int(r.get("actions") or 0) for r in self._rows),\n        }\n\n    def write_summary(\n        self, scored_environments: int, summary_path: str | Path | None = None\n    ) -> dict[str, Any]:\n        summary = self.coverage_summary(scored_environments)\n        out = Path(summary_path) if summary_path else self.path.with_suffix(".summary.json")\n        out.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n        return summary\n\n\nclass GameTimer:\n    def __init__(self) -> None:\n        self._t0 = time.monotonic()\n\n    def elapsed(self) -> float:\n        return time.monotonic() - self._t0\n\n\ndef load_jsonl(path: str | Path) -> list[dict[str, Any]]:\n    rows: list[dict[str, Any]] = []\n    for line in Path(path).read_text(encoding="utf-8").splitlines():\n        line = line.strip()\n        if line:\n            rows.append(json.loads(line))\n    return rows\n', encoding='utf-8')
(_ARC3_PKG / 'portfolio.py').write_text('"""Open-loop portfolio scheduler for competition-mode runs.\n\nCompetition mode: one `make` per environment, no mid-run scorecard, untouched\ngames score 0 and still divide the mean. Time left on a stuck game is often\nworth more as a first-level attempt elsewhere — or as a revisit on a progressing game.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom enum import Enum\n\n\nclass Phase(str, Enum):\n    EXPLORE = "explore"  # first touch: try to clear level 1\n    EXPLOIT = "exploit"  # continue a game that already cleared >=1\n    REVISIT = "revisit"\n\n\n@dataclass\nclass GameBudget:\n    game_id: str\n    allocated_seconds: float\n    phase: Phase = Phase.EXPLORE\n    levels_completed: int = 0\n    spent_seconds: float = 0.0\n    abandon_reason: str = ""\n\n    @property\n    def remaining_seconds(self) -> float:\n        return max(0.0, self.allocated_seconds - self.spent_seconds)\n\n    @property\n    def exhausted(self) -> bool:\n        return self.remaining_seconds <= 1e-9 or bool(self.abandon_reason)\n\n\n@dataclass\nclass PortfolioPlan:\n    total_seconds: float\n    scored_environments: int\n    games: list[GameBudget] = field(default_factory=list)\n    reserve_seconds: float = 0.0\n\n    def next_game(self) -> GameBudget | None:\n        for game in self.games:\n            if not game.exhausted:\n                return game\n        return None\n\n\ndef build_equal_explore_plan(\n    game_ids: list[str],\n    total_seconds: float,\n    *,\n    scored_environments: int | None = None,\n    reserve_fraction: float = 0.08,\n    min_seconds_per_game: float = 90.0,\n) -> PortfolioPlan:\n    """First pass: equal explore budget; hold a reserve for exploit revisits."""\n    if total_seconds <= 0:\n        raise ValueError("total_seconds must be positive")\n    if not game_ids:\n        raise ValueError("game_ids must be non-empty")\n    scored = scored_environments if scored_environments is not None else len(game_ids)\n    reserve = total_seconds * reserve_fraction\n    pool = total_seconds - reserve\n    per = pool / len(game_ids)\n    if per < min_seconds_per_game:\n        # Prefer covering fewer games with a usable slice over touching all with noise.\n        max_games = max(1, int(pool // min_seconds_per_game))\n        chosen = game_ids[:max_games]\n        per = pool / len(chosen)\n        game_ids = chosen\n    games = [GameBudget(game_id=g, allocated_seconds=per, phase=Phase.EXPLORE) for g in game_ids]\n    return PortfolioPlan(\n        total_seconds=total_seconds,\n        scored_environments=scored,\n        games=games,\n        reserve_seconds=reserve,\n    )\n\n\ndef should_abandon(\n    *,\n    spent_seconds: float,\n    allocated_seconds: float,\n    levels_completed: int,\n    actions: int,\n    no_level_progress_seconds: float,\n    min_actions_before_abandon: int = 15,\n) -> str:\n    """Return abandon reason or empty string to continue.\n\n    Hard stop when the slice is gone. Soft abandon when we burned most of the\n    slice with zero levels and enough actions to prove we are stuck.\n    """\n    if spent_seconds >= allocated_seconds:\n        return "time_budget_exhausted"\n    if (\n        levels_completed == 0\n        and actions >= min_actions_before_abandon\n        and no_level_progress_seconds >= 0.7 * allocated_seconds\n    ):\n        return "no_level1_progress"\n    return ""\n\n\ndef reallocate_on_level(\n    plan: PortfolioPlan,\n    game_id: str,\n    *,\n    bonus_seconds: float,\n) -> None:\n    """Move reserve time onto a game that just cleared a level."""\n    target = next((g for g in plan.games if g.game_id == game_id), None)\n    if target is None:\n        return\n    take = min(plan.reserve_seconds, max(0.0, bonus_seconds))\n    plan.reserve_seconds -= take\n    target.allocated_seconds += take\n    target.phase = Phase.EXPLOIT\n    target.levels_completed = max(target.levels_completed, 1)\n\n\ndef mark_spent(plan: PortfolioPlan, game_id: str, spent_seconds: float, levels_completed: int, abandon_reason: str = "") -> None:\n    game = next((g for g in plan.games if g.game_id == game_id), None)\n    if game is None:\n        return\n    game.spent_seconds += spent_seconds\n    game.levels_completed = max(game.levels_completed, levels_completed)\n    if abandon_reason:\n        game.abandon_reason = abandon_reason\n    if game.levels_completed > 0 and game.phase == Phase.EXPLORE:\n        game.phase = Phase.EXPLOIT\n', encoding='utf-8')
(_ARC3_PKG / 'portfolio_loop.py').write_text('"""Drive a multi-game run from a PortfolioPlan.\n\nShared scheduler for ``arc3.evaluate`` and the Qwen/Duck budget bridge.\n``play_fn`` is injected so unit tests stay free of Duck and the game engine.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Any, Callable\n\nfrom .instrumentation import GameTrace, TraceRecorder\nfrom .portfolio import (\n    PortfolioPlan,\n    build_equal_explore_plan,\n    mark_spent,\n    reallocate_on_level,\n    should_abandon,\n)\n\n\n@dataclass\nclass PlayOutcome:\n    """Minimal result a play callback must return for portfolio accounting."""\n\n    game_id: str\n    wall_seconds: float\n    actions: int\n    levels_completed: int\n    level_indices_completed: list[int] = field(default_factory=list)\n    abandon_reason: str = ""\n    tokens_in: int = 0\n    tokens_out: int = 0\n    llm_calls: int = 0\n    score: float | None = None\n    meta: dict[str, Any] = field(default_factory=dict)\n\n\nPlayFn = Callable[[str, float], PlayOutcome]\n\n\n@dataclass\nclass PortfolioRunResult:\n    outcomes: list[PlayOutcome]\n    plan: PortfolioPlan\n    recorder: TraceRecorder | None = None\n\n\ndef run_portfolio(\n    game_ids: list[str],\n    play_fn: PlayFn,\n    *,\n    total_seconds: float,\n    scored_environments: int | None = None,\n    reserve_fraction: float = 0.08,\n    min_seconds_per_game: float = 90.0,\n    bonus_seconds_on_level: float = 120.0,\n    min_actions_before_abandon: int = 15,\n    trace_path: str | None = None,\n    on_outcome: Callable[[PlayOutcome], None] | None = None,\n) -> PortfolioRunResult:\n    """Visit games under an equal-explore plan until budgets are exhausted."""\n    plan = build_equal_explore_plan(\n        list(game_ids),\n        total_seconds,\n        scored_environments=scored_environments,\n        reserve_fraction=reserve_fraction,\n        min_seconds_per_game=min_seconds_per_game,\n    )\n    recorder = TraceRecorder(trace_path) if trace_path else None\n    outcomes: list[PlayOutcome] = []\n\n    while True:\n        budget = plan.next_game()\n        if budget is None:\n            break\n        max_seconds = budget.remaining_seconds\n        if max_seconds <= 1e-9:\n            mark_spent(\n                plan,\n                budget.game_id,\n                spent_seconds=0.0,\n                levels_completed=budget.levels_completed,\n                abandon_reason="time_budget_exhausted",\n            )\n            continue\n\n        outcome = play_fn(budget.game_id, max_seconds)\n        reason = outcome.abandon_reason or should_abandon(\n            spent_seconds=outcome.wall_seconds,\n            allocated_seconds=budget.allocated_seconds,\n            levels_completed=outcome.levels_completed,\n            actions=outcome.actions,\n            no_level_progress_seconds=outcome.wall_seconds,\n            min_actions_before_abandon=min_actions_before_abandon,\n        )\n        if reason and not outcome.abandon_reason:\n            outcome.abandon_reason = reason\n\n        if outcome.levels_completed > budget.levels_completed:\n            reallocate_on_level(\n                plan,\n                budget.game_id,\n                bonus_seconds=bonus_seconds_on_level\n                * max(1, outcome.levels_completed - budget.levels_completed),\n            )\n\n        mark_spent(\n            plan,\n            budget.game_id,\n            spent_seconds=outcome.wall_seconds,\n            levels_completed=outcome.levels_completed,\n            abandon_reason=outcome.abandon_reason,\n        )\n        outcomes.append(outcome)\n        if on_outcome is not None:\n            on_outcome(outcome)\n        if recorder is not None:\n            recorder.record(\n                GameTrace(\n                    game_id=outcome.game_id,\n                    wall_seconds=outcome.wall_seconds,\n                    actions=outcome.actions,\n                    levels_completed=outcome.levels_completed,\n                    level_indices_completed=list(outcome.level_indices_completed),\n                    abandon_reason=outcome.abandon_reason,\n                    tokens_in=outcome.tokens_in,\n                    tokens_out=outcome.tokens_out,\n                    llm_calls=outcome.llm_calls,\n                    score=outcome.score,\n                    meta=dict(outcome.meta),\n                )\n            )\n\n    if recorder is not None:\n        recorder.write_summary(plan.scored_environments)\n    return PortfolioRunResult(outcomes=outcomes, plan=plan, recorder=recorder)\n', encoding='utf-8')
(_ARC3_PKG / 'qwen_portfolio.py').write_text('"""Bridge portfolio + JSONL tracing onto the Qwen/Duck solver surface.\n\nDuck Gate-1 notebooks drive per-game wall via duck-typed solver attributes:\n\n* ``solver.max_runtime_s_per_game`` — per-game wall budget\n* ``solver.analyzer_timeout`` — single LLM call timeout (left alone here)\n\nSession/progress rows typically carry ``game_id``, ``levels_completed``,\n``actions_taken``, ``active_wall_seconds``, ``llm_calls``, ``generated_tokens``.\n\nThis module never imports Duck ``inference``, so local coverage gates stay\nGPU-free. Notebooks add the repo root to ``sys.path`` and call the helpers below.\n\nExpect-queue is intentionally *not* wired into the Qwen XML/tool loop — Duck\nalready executes tool plans; grafting Retrodict-style board expects onto model\ntool XML would break the live path. Use ``arc3.expect_queue`` only for non-Duck\nlocal agents that emit ``QueuedAction`` plans.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass\nfrom typing import Any, Mapping\n\nfrom .instrumentation import GameTrace, TraceRecorder\nfrom .portfolio import (\n    GameBudget,\n    PortfolioPlan,\n    build_equal_explore_plan,\n    mark_spent,\n    reallocate_on_level,\n    should_abandon,\n)\nfrom .portfolio_loop import PlayOutcome, run_portfolio\n\nENV_TRACE = "ARC3_TRACE"\nENV_PORTFOLIO_SECONDS = "ARC3_PORTFOLIO_SECONDS"\nENV_SCORED_ENVIRONMENTS = "ARC3_SCORED_ENVIRONMENTS"\nENV_MIN_SECONDS_PER_GAME = "ARC3_MIN_SECONDS_PER_GAME"\nENV_RESERVE_FRACTION = "ARC3_RESERVE_FRACTION"\nENV_BONUS_ON_LEVEL = "ARC3_BONUS_SECONDS_ON_LEVEL"\n\n\ndef resolve_trace_path(cli_value: str | None = None) -> str | None:\n    """CLI ``--trace`` wins; else non-empty ``ARC3_TRACE``."""\n    if cli_value:\n        return cli_value\n    env = os.environ.get(ENV_TRACE, "").strip()\n    return env or None\n\n\ndef resolve_portfolio_seconds(cli_value: float | None = None) -> float | None:\n    if cli_value is not None and cli_value > 0:\n        return float(cli_value)\n    raw = os.environ.get(ENV_PORTFOLIO_SECONDS, "").strip()\n    if not raw:\n        return None\n    return float(raw)\n\n\ndef resolve_float_env(name: str, default: float) -> float:\n    raw = os.environ.get(name, "").strip()\n    return float(raw) if raw else default\n\n\ndef resolve_int_env(name: str, default: int) -> int:\n    raw = os.environ.get(name, "").strip()\n    return int(raw) if raw else default\n\n\n@dataclass\nclass DuckBudgetSlice:\n    """One portfolio assignment expressed in Duck solver knobs."""\n\n    game_id: str\n    max_runtime_s_per_game: float\n    phase: str\n    allocated_seconds: float\n    remaining_seconds: float\n\n\ndef plan_for_qwen(\n    game_ids: list[str],\n    total_seconds: float,\n    *,\n    scored_environments: int | None = None,\n    reserve_fraction: float | None = None,\n    min_seconds_per_game: float | None = None,\n) -> PortfolioPlan:\n    return build_equal_explore_plan(\n        game_ids,\n        total_seconds,\n        scored_environments=scored_environments\n        if scored_environments is not None\n        else resolve_int_env(ENV_SCORED_ENVIRONMENTS, len(game_ids)),\n        reserve_fraction=reserve_fraction\n        if reserve_fraction is not None\n        else resolve_float_env(ENV_RESERVE_FRACTION, 0.08),\n        min_seconds_per_game=min_seconds_per_game\n        if min_seconds_per_game is not None\n        else resolve_float_env(ENV_MIN_SECONDS_PER_GAME, 90.0),\n    )\n\n\ndef duck_slices(plan: PortfolioPlan) -> list[DuckBudgetSlice]:\n    return [\n        DuckBudgetSlice(\n            game_id=g.game_id,\n            max_runtime_s_per_game=g.remaining_seconds,\n            phase=g.phase.value if hasattr(g.phase, "value") else str(g.phase),\n            allocated_seconds=g.allocated_seconds,\n            remaining_seconds=g.remaining_seconds,\n        )\n        for g in plan.games\n        if not g.exhausted\n    ]\n\n\ndef apply_budget_to_solver(solver: Any, slice_: DuckBudgetSlice | GameBudget) -> float:\n    """Set Duck ``max_runtime_s_per_game`` from a portfolio slice. Returns seconds applied."""\n    if isinstance(slice_, GameBudget):\n        seconds = float(slice_.remaining_seconds)\n    else:\n        seconds = float(slice_.max_runtime_s_per_game)\n    solver.max_runtime_s_per_game = seconds\n    return seconds\n\n\ndef apply_plan_game_to_solver(solver: Any, plan: PortfolioPlan, game_id: str) -> float:\n    game = next((g for g in plan.games if g.game_id == game_id), None)\n    if game is None:\n        raise KeyError(f"game_id {game_id!r} not in portfolio plan")\n    return apply_budget_to_solver(solver, game)\n\n\ndef trace_from_duck_row(row: Mapping[str, Any], *, abandon_reason: str = "") -> GameTrace:\n    """Convert a Gate-1 progress / session row into a coverage GameTrace."""\n    levels = int(row.get("levels_completed") or 0)\n    actions = int(row.get("actions_taken") or row.get("actions") or 0)\n    wall = float(\n        row.get("active_wall_seconds")\n        or row.get("wall_seconds")\n        or row.get("persisted_elapsed_seconds")\n        or 0.0\n    )\n    indices = row.get("level_indices_completed")\n    if indices is None and levels > 0:\n        indices = list(range(1, levels + 1))\n    reason = abandon_reason or str(row.get("abandon_reason") or "")\n    if not reason and levels == 0 and actions > 0:\n        state = str(row.get("state") or "")\n        if state in {"gave_up", "cancelled"}:\n            reason = f"duck_{state}"\n    return GameTrace(\n        game_id=str(row.get("game_id") or ""),\n        wall_seconds=wall,\n        actions=actions,\n        levels_completed=levels,\n        level_indices_completed=list(indices or []),\n        abandon_reason=reason,\n        tokens_in=int(row.get("tokens_in") or 0),\n        tokens_out=int(row.get("generated_tokens") or row.get("tokens_out") or 0),\n        llm_calls=int(row.get("llm_calls") or 0),\n        score=float(row["final_score"]) if row.get("final_score") is not None else None,\n        meta={\n            k: row[k]\n            for k in ("state", "number_of_levels", "actions_per_level")\n            if k in row\n        },\n    )\n\n\ndef record_duck_rows(\n    rows: list[Mapping[str, Any]],\n    *,\n    trace_path: str,\n    scored_environments: int,\n) -> dict[str, Any]:\n    recorder = TraceRecorder(trace_path)\n    for row in rows:\n        recorder.record(trace_from_duck_row(row))\n    return recorder.write_summary(scored_environments)\n\n\ndef notify_level_and_spend(\n    plan: PortfolioPlan,\n    *,\n    game_id: str,\n    wall_seconds: float,\n    levels_completed: int,\n    actions: int,\n    bonus_seconds: float | None = None,\n) -> str:\n    """Update plan after one Duck game finishes; return abandon reason (may be empty)."""\n    game = next((g for g in plan.games if g.game_id == game_id), None)\n    prev_levels = game.levels_completed if game else 0\n    if levels_completed > prev_levels:\n        per = (\n            bonus_seconds\n            if bonus_seconds is not None\n            else resolve_float_env(ENV_BONUS_ON_LEVEL, 120.0)\n        )\n        reallocate_on_level(\n            plan,\n            game_id,\n            bonus_seconds=per * max(1, levels_completed - prev_levels),\n        )\n    allocated = game.allocated_seconds if game else wall_seconds\n    reason = should_abandon(\n        spent_seconds=wall_seconds,\n        allocated_seconds=allocated,\n        levels_completed=levels_completed,\n        actions=actions,\n        no_level_progress_seconds=wall_seconds,\n    )\n    mark_spent(\n        plan,\n        game_id,\n        spent_seconds=wall_seconds,\n        levels_completed=levels_completed,\n        abandon_reason=reason,\n    )\n    return reason\n\n\ndef dry_run_budget_api(\n    game_ids: list[str],\n    total_seconds: float,\n    *,\n    play_seconds: float = 10.0,\n    levels_by_game: Mapping[str, int] | None = None,\n) -> tuple[PortfolioPlan, list[PlayOutcome]]:\n    """GPU-free harness: fake plays that consume portfolio time via the budget API."""\n\n    class _Solver:\n        max_runtime_s_per_game = 0.0\n\n    levels_by_game = dict(levels_by_game or {})\n\n    def play_fn(game_id: str, max_seconds: float) -> PlayOutcome:\n        solver = _Solver()\n        applied = apply_budget_to_solver(\n            solver,\n            DuckBudgetSlice(\n                game_id=game_id,\n                max_runtime_s_per_game=max_seconds,\n                phase="explore",\n                allocated_seconds=max_seconds,\n                remaining_seconds=max_seconds,\n            ),\n        )\n        assert abs(applied - max_seconds) < 1e-9\n        assert solver.max_runtime_s_per_game == max_seconds\n        # Consume the full portfolio slice (Duck applies one session budget per visit).\n        spent = max_seconds\n        levels = int(levels_by_game.get(game_id, 0))\n        actions = max(15, int(min(play_seconds, max_seconds)))\n        return PlayOutcome(\n            game_id=game_id,\n            wall_seconds=spent,\n            actions=actions,\n            levels_completed=levels,\n            level_indices_completed=list(range(1, levels + 1)),\n            meta={"solver_budget": solver.max_runtime_s_per_game},\n        )\n\n    result = run_portfolio(\n        game_ids,\n        play_fn,\n        total_seconds=total_seconds,\n        min_seconds_per_game=min(90.0, total_seconds / max(1, len(game_ids))),\n        bonus_seconds_on_level=30.0,\n    )\n    return result.plan, result.outcomes\n\n\n@dataclass\nclass SequentialApplyRecord:\n    """One apply → play → session-end notify cycle."""\n\n    game_id: str\n    applied_seconds: float\n    wall_seconds: float\n    levels_completed: int\n    actions: int\n    abandon_reason: str\n    allocated_after: float\n    remaining_after: float\n    reserve_after: float\n\n\ndef live_reallocate_if_level_up(\n    plan: PortfolioPlan,\n    solver: Any,\n    *,\n    game_id: str,\n    levels_completed: int,\n    bonus_seconds: float | None = None,\n) -> float:\n    """Mid-session: move reserve onto a newly cleared level and bump Duck wall.\n\n    Does **not** mark spend — call :func:`notify_level_and_spend` at session end.\n    Returns the solver ``max_runtime_s_per_game`` after the bump, or ``0.0`` if\n    no new levels were observed.\n    """\n    game = next((g for g in plan.games if g.game_id == game_id), None)\n    if game is None:\n        return 0.0\n    prev_levels = int(game.levels_completed)\n    if levels_completed <= prev_levels:\n        return 0.0\n    per = (\n        bonus_seconds\n        if bonus_seconds is not None\n        else resolve_float_env(ENV_BONUS_ON_LEVEL, 120.0)\n    )\n    reallocate_on_level(\n        plan,\n        game_id,\n        bonus_seconds=per * max(1, levels_completed - prev_levels),\n    )\n    game.levels_completed = max(game.levels_completed, levels_completed)\n    return apply_budget_to_solver(solver, game)\n\n\ndef notify_session_end(\n    plan: PortfolioPlan,\n    *,\n    game_id: str,\n    wall_seconds: float,\n    levels_completed: int,\n    actions: int,\n    bonus_seconds: float | None = None,\n    already_live_reallocated: bool = False,\n) -> str:\n    """Session-end portfolio update (alias with explicit live-path semantics).\n\n    If mid-session :func:`live_reallocate_if_level_up` already moved reserve for\n    these levels, pass ``already_live_reallocated=True`` so we only mark spend /\n    abandon (levels on the plan already reflect the bonus).\n    """\n    if already_live_reallocated:\n        game = next((g for g in plan.games if g.game_id == game_id), None)\n        allocated = game.allocated_seconds if game else wall_seconds\n        reason = should_abandon(\n            spent_seconds=wall_seconds,\n            allocated_seconds=allocated,\n            levels_completed=levels_completed,\n            actions=actions,\n            no_level_progress_seconds=wall_seconds,\n        )\n        mark_spent(\n            plan,\n            game_id,\n            spent_seconds=wall_seconds,\n            levels_completed=levels_completed,\n            abandon_reason=reason,\n        )\n        return reason\n    return notify_level_and_spend(\n        plan,\n        game_id=game_id,\n        wall_seconds=wall_seconds,\n        levels_completed=levels_completed,\n        actions=actions,\n        bonus_seconds=bonus_seconds,\n    )\n\n\ndef drive_sequential_portfolio(\n    solver: Any,\n    plan: PortfolioPlan,\n    play_session: Any,\n    *,\n    game_ids: list[str] | None = None,\n    bonus_seconds: float | None = None,\n    max_visits: int | None = None,\n) -> list[SequentialApplyRecord]:\n    """Apply → play → session-end notify, repeating via ``plan.next_game()``.\n\n    Notify runs **before** the next ``apply_plan_game_to_solver``, so a\n    level-triggered reserve move changes the subsequent allocation (typically\n    a revisit with leftover exploit budget).\n\n    ``play_session(game_id, applied_seconds) -> Mapping`` must return keys\n    ``levels_completed``, ``actions`` / ``actions_taken``, and\n    ``wall_seconds`` / ``active_wall_seconds``.\n    """\n    records: list[SequentialApplyRecord] = []\n    visits = 0\n    fixed = list(game_ids) if game_ids is not None else None\n    fixed_idx = 0\n\n    while True:\n        if max_visits is not None and visits >= max_visits:\n            break\n        if fixed is not None:\n            if fixed_idx >= len(fixed):\n                break\n            game_id = fixed[fixed_idx]\n            fixed_idx += 1\n            game = next((g for g in plan.games if g.game_id == game_id), None)\n            if game is None or game.exhausted:\n                continue\n        else:\n            game = plan.next_game()\n            if game is None:\n                break\n            game_id = game.game_id\n\n        applied = apply_plan_game_to_solver(solver, plan, game_id)\n        row = dict(play_session(game_id, applied) or {})\n        levels = int(row.get("levels_completed") or 0)\n        actions = int(row.get("actions_taken") or row.get("actions") or 0)\n        wall = float(\n            row.get("active_wall_seconds")\n            or row.get("wall_seconds")\n            or applied\n        )\n        reason = notify_level_and_spend(\n            plan,\n            game_id=game_id,\n            wall_seconds=wall,\n            levels_completed=levels,\n            actions=actions,\n            bonus_seconds=bonus_seconds,\n        )\n        game_after = next(g for g in plan.games if g.game_id == game_id)\n        records.append(\n            SequentialApplyRecord(\n                game_id=game_id,\n                applied_seconds=applied,\n                wall_seconds=wall,\n                levels_completed=levels,\n                actions=actions,\n                abandon_reason=reason,\n                allocated_after=game_after.allocated_seconds,\n                remaining_after=game_after.remaining_seconds,\n                reserve_after=plan.reserve_seconds,\n            )\n        )\n        visits += 1\n    return records\n', encoding='utf-8')
if str(WORKING_DIR) not in _arc3_sys.path:
    _arc3_sys.path.insert(0, str(WORKING_DIR))
from arc3.qwen_portfolio import (  # noqa: E402
    plan_for_qwen,
    apply_plan_game_to_solver,
    live_reallocate_if_level_up,
    notify_session_end,
    notify_level_and_spend,
    record_duck_rows,
)
print('ARC3_PORTFOLIO_HELPERS_READY', flush=True)


In [ ]:
# Gate 1 production path with a minimum-scale lifecycle smoke switch.
GATE1_SMOKE_MODE = True
GATE1_SMOKE_GAME_SECONDS = 900.0
GATE1_SMOKE_NOTEBOOK_LIMIT_SECONDS = 7200.0
GATE1_SMOKE_GAME_COUNT = 6
PORTFOLIO_SMOKE_SECONDS = float(os.environ['ARC3_PORTFOLIO_SECONDS'])
GATE1_WAVE_CAP_SECONDS = 6600.0
GATE1_SAFETY_MARGIN_SECONDS = 4860.0
GATE1_SOFT_STOP_GRACE_SECONDS = 120.0

bm.solver.max_runtime_s_per_game = (
    GATE1_SMOKE_GAME_SECONDS if GATE1_SMOKE_MODE else GATE1_WAVE_CAP_SECONDS
)
bm.solver.analyzer_timeout = 60.0 if GATE1_SMOKE_MODE else 900.0
bm.solver.concurrency = 1  # portfolio sequential: never parallel apply before notify
if not GATE1_SMOKE_MODE:
    print("PORTFOLIO_CONCURRENCY_FORCE concurrency=1 (was wave default 28)", flush=True)
bm.solver.max_actions_per_game = 400
bm.solver.save_request_logs = False

if float(getattr(target, "max_runtime_s", 0.0) or 0.0) != 32400.0:
    raise RuntimeError(
        f"Expected the 32400-second notebook budget, got {target.max_runtime_s!r}."
    )
if GATE1_SAFETY_MARGIN_SECONDS < 0.15 * float(target.max_runtime_s):
    raise RuntimeError("Gate 1 safety margin is below 15% of the notebook budget.")

import threading as _gate1_threading
from inference.agent.tool_agent import ToolAgent as _Gate1ToolAgent
from inference.framework.solver import _HarnessGameSession as _Gate1Session

_GATE1_METRICS_LOCK = _gate1_threading.Lock()
_GATE1_METRICS = {
    "instrumentation_epoch": time.time(),
    "first_request_started_epoch": None,
    "first_response_epoch": None,
    "llm_calls_total": 0,
}
_GATE1_SESSION_METRICS = {}

if not getattr(_Gate1ToolAgent, "_gate1_call_counter_installed", False):
    _GATE1_ORIGINAL_CHAT_COMPLETION = _Gate1ToolAgent._chat_completion

    def _gate1_counted_chat_completion(self, *args, **kwargs):
        is_first = False
        started_epoch = time.time()
        with _GATE1_METRICS_LOCK:
            self._gate1_llm_calls = int(getattr(self, "_gate1_llm_calls", 0)) + 1
            _GATE1_METRICS["llm_calls_total"] += 1
            if _GATE1_METRICS["first_request_started_epoch"] is None:
                _GATE1_METRICS["first_request_started_epoch"] = started_epoch
                is_first = True
        try:
            return _GATE1_ORIGINAL_CHAT_COMPLETION(self, *args, **kwargs)
        finally:
            if is_first:
                with _GATE1_METRICS_LOCK:
                    _GATE1_METRICS["first_response_epoch"] = time.time()

    _Gate1ToolAgent._chat_completion = _gate1_counted_chat_completion
    _Gate1ToolAgent._gate1_call_counter_installed = True

if not getattr(_Gate1Session, "_gate1_session_timer_installed", False):
    _GATE1_ORIGINAL_SESSION_PLAY = _Gate1Session.play

    def _gate1_traced_session_play(self):
        active_started = time.monotonic()
        run = getattr(self.game, "game_run", None)
        game_id = getattr(run, "game_id", str(self.game_index))
        live_bumped = False
        try:
            if "_PORTFOLIO_PLAN" in globals() and _PORTFOLIO_PLAN is not None:
                applied = apply_plan_game_to_solver(bm.solver, _PORTFOLIO_PLAN, game_id)
                print(
                    f'PORTFOLIO_APPLY game_id={game_id} max_runtime_s_per_game={applied}',
                    flush=True,
                )
            return _GATE1_ORIGINAL_SESSION_PLAY(self)
        finally:
            wall = time.monotonic() - active_started
            run_end = getattr(self.game, "game_run", None) or run
            levels = int(getattr(run_end, "levels_completed", 0) or 0) if run_end is not None else 0
            history = getattr(run_end, "history", None) if run_end is not None else None
            actions = len(history) if history is not None else 0
            abandon_reason = ""
            if "_PORTFOLIO_PLAN" in globals() and _PORTFOLIO_PLAN is not None:
                bumped = live_reallocate_if_level_up(
                    _PORTFOLIO_PLAN,
                    bm.solver,
                    game_id=game_id,
                    levels_completed=levels,
                )
                if bumped:
                    live_bumped = True
                    print(
                        f'PORTFOLIO_LIVE_REALLOC game_id={game_id} '
                        f'levels={levels} max_runtime_s_per_game={bumped}',
                        flush=True,
                    )
                notified = globals().setdefault('_PORTFOLIO_SESSION_NOTIFIED', set())
                if game_id not in notified:
                    abandon_reason = notify_session_end(
                        _PORTFOLIO_PLAN,
                        game_id=game_id,
                        wall_seconds=float(wall),
                        levels_completed=levels,
                        actions=int(actions),
                        already_live_reallocated=live_bumped,
                    )
                    notified.add(game_id)
                    print(
                        'PORTFOLIO_SESSION_NOTIFY '
                        + json.dumps(
                            {
                                'game_id': game_id,
                                'wall': wall,
                                'levels': levels,
                                'actions': actions,
                                'abandon_reason': abandon_reason,
                                'live_bumped': live_bumped,
                            },
                            sort_keys=True,
                        ),
                        flush=True,
                    )
            row = {
                "session_started": True,
                "active_wall_seconds": wall,
                "llm_calls": int(getattr(self.analyzer, "_gate1_llm_calls", 0)),
                "abandon_reason": abandon_reason,
                "levels_completed": levels,
                "actions_taken": int(actions),
            }
            with _GATE1_METRICS_LOCK:
                _GATE1_SESSION_METRICS[game_id] = row

    _Gate1Session.play = _gate1_traced_session_play
    _Gate1Session._gate1_session_timer_installed = True

print(
    f"GATE1_SETTINGS smoke={GATE1_SMOKE_MODE} "
    f"budget_s={bm.solver.max_runtime_s_per_game} concurrency={bm.solver.concurrency} "
    f"analyzer_timeout={bm.solver.analyzer_timeout} "
    f"action_cap={bm.solver.max_actions_per_game} request_logs={bm.solver.save_request_logs} "
    f"safety_margin_s={GATE1_SAFETY_MARGIN_SECONDS} "
    f"soft_stop_grace_s={GATE1_SOFT_STOP_GRACE_SECONDS}",
    flush=True,
)


In [ ]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


def _offline_games(env_dir: str):
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

PUBLIC_GAME_IDS = (
    "tn36-ef4dde99", "lf52-271a04aa", "cn04-2fe56bfb", "bp35-0a0ad940",
    "wa30-ee6fef47", "lp85-305b61c3", "r11l-495a7899", "tu93-0768757b",
    "sp80-589a99af", "m0r0-492f87ba", "vc33-5430563c", "ar25-0c556536",
    "ka59-38d34dbb", "sc25-635fd71a", "sk48-d8078629", "dc22-fdcac232",
    "cd82-fb555c5d", "ft09-0d8bbf25", "g50t-5849a774", "ls20-9607627b",
    "re86-8af5384d", "s5i5-18d95033", "sb26-7fbdac44", "su15-1944f8ab",
    "tr87-cd924810",
)

if TRUE_SUBMISSION and GATE1_SMOKE_MODE:
    raise RuntimeError("Smoke mode must be disabled before a competition submission.")

if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
    _gate1_expected_ids = [game.env_name for game in bm.games]
else:
    competition_env_files = str(
        Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent
        / "environment_files"
    )
    offline_games = _offline_games(competition_env_files)
    offline_by_id = {game.env_name: game for game in offline_games}
    if len(offline_by_id) != len(offline_games):
        raise RuntimeError("The offline public game list contains duplicate IDs.")
    missing = sorted(set(PUBLIC_GAME_IDS) - set(offline_by_id))
    extra = sorted(set(offline_by_id) - set(PUBLIC_GAME_IDS))
    if missing or extra:
        raise RuntimeError(f"Offline public game set changed; missing={missing}, extra={extra}.")
    _gate1_expected_ids = list(
        PUBLIC_GAME_IDS[:GATE1_SMOKE_GAME_COUNT] if GATE1_SMOKE_MODE else PUBLIC_GAME_IDS
    )
    bm.games = [offline_by_id[game_id] for game_id in _gate1_expected_ids]
    print(
        f"PUBLIC_SELECTION smoke={GATE1_SMOKE_MODE} games={len(bm.games)} passes=1",
        flush=True,
    )

_PORTFOLIO_SESSION_NOTIFIED = set()
_PORTFOLIO_PLAN = plan_for_qwen(
    list(_gate1_expected_ids),
    float(PORTFOLIO_SMOKE_SECONDS),
    scored_environments=int(os.environ.get('ARC3_SCORED_ENVIRONMENTS', '25')),
    min_seconds_per_game=float(os.environ.get('ARC3_MIN_SECONDS_PER_GAME', '90')),
)
_portfolio_plan_path = WORKING_DIR / 'portfolio_plan.json'
_portfolio_plan_path.write_text(
    json.dumps(
        {
            'total_seconds': _PORTFOLIO_PLAN.total_seconds,
            'scored_environments': _PORTFOLIO_PLAN.scored_environments,
            'reserve_seconds': _PORTFOLIO_PLAN.reserve_seconds,
            'games': [
                {
                    'game_id': g.game_id,
                    'allocated_seconds': g.allocated_seconds,
                    'phase': g.phase.value if hasattr(g.phase, 'value') else str(g.phase),
                }
                for g in _PORTFOLIO_PLAN.games
            ],
        },
        indent=2,
        sort_keys=True,
    )
    + '\n',
    encoding='utf-8',
)
print(
    'PORTFOLIO_PLAN '
    + json.dumps(
        {
            'games': len(_PORTFOLIO_PLAN.games),
            'total_seconds': _PORTFOLIO_PLAN.total_seconds,
            'reserve_seconds': _PORTFOLIO_PLAN.reserve_seconds,
            'path': str(_portfolio_plan_path),
        },
        sort_keys=True,
    ),
    flush=True,
)

bm.n_passes = 1
bm.game_weights = None

import asyncio as _gate1_asyncio
import os as _gate1_os
import signal as _gate1_signal

_GATE1_PROGRESS_DIR = WORKING_DIR / "gate1-progress"
_GATE1_PROGRESS_DIR.mkdir(parents=True, exist_ok=True)
_GATE1_PROGRESS_START = time.monotonic()
_gate1_hard_guard_triggered = False
_gate1_benchmark_error = None
_gate1_benchmark_ok = False
_gate1_teardown_error = None
_gate1_teardown_ok = False
_gate1_teardown_attempts = []
_gate1_teardown_result = {}
_gate1_post_gpu_rows = []


def _gate1_atomic_json(path, value):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")
    _gate1_os.replace(tmp, path)


def _gate1_run_snapshot(run):
    session = _GATE1_SESSION_METRICS.get(run.game_id) or {}
    generated_tokens = sum(int(record.generated_tokens) for record in run.history)
    generated_tokens += int(getattr(run, "final_generated_tokens", 0) or 0)
    base_raw = run.base_actions_per_level
    return {
        "game_id": run.game_id,
        "state": str(run.state),
        "final_score": float(run.final_score) if run.final_score is not None else None,
        "actions_taken": len(run.history),
        "actions_per_level": [int(value) for value in run.actions_per_level],
        "base_actions_per_level": (
            None if base_raw is None else [int(value) for value in base_raw]
        ),
        "levels_completed": int(run.levels_completed),
        "number_of_levels": int(run.number_of_levels),
        "generated_tokens": generated_tokens,
        "llm_calls": int(session.get("llm_calls", 0)),
        "active_wall_seconds": session.get("active_wall_seconds"),
        "persisted_elapsed_seconds": time.monotonic() - _GATE1_PROGRESS_START,
    }


def _gate1_persist_run(run, terminal):
    row = _gate1_run_snapshot(run)
    row["terminal"] = bool(terminal)
    _gate1_atomic_json(_GATE1_PROGRESS_DIR / f"{run.game_id}.json", row)
    return row


def _gate1_write_progress_index():
    rows = []
    for path in sorted(_GATE1_PROGRESS_DIR.glob("*.json")):
        try:
            rows.append(json.loads(path.read_text()))
        except Exception as exc:
            print(f"GATE1_PROGRESS_READ_ERROR path={path} error={exc!r}", flush=True)
    _gate1_atomic_json(WORKING_DIR / "gate1_progress.json", rows)
    jsonl_tmp = WORKING_DIR / "gate1_progress.jsonl.tmp"
    jsonl_tmp.write_text("".join(json.dumps(row, sort_keys=True) + "\n" for row in rows))
    _gate1_os.replace(jsonl_tmp, WORKING_DIR / "gate1_progress.jsonl")
    return rows


def _gate1_flush_all_runs():
    for run in list(bm.game_runs):
        terminal = str(run.state) != "playing"
        _gate1_persist_run(run, terminal=terminal)
    rows = _gate1_write_progress_index()
    try:
        bm._save_json()
    except Exception as exc:
        print(f"GATE1_BENCHMARK_SAVE_ERROR {exc!r}", flush=True)
    return rows


async def _gate1_progress_loop(stop_event):
    emitted = set()
    try:
        while not stop_event.is_set():
            for run in list(bm.game_runs):
                if (
                    "_PORTFOLIO_PLAN" in globals()
                    and _PORTFOLIO_PLAN is not None
                    and str(run.state) == "playing"
                ):
                    _live = live_reallocate_if_level_up(
                        _PORTFOLIO_PLAN,
                        bm.solver,
                        game_id=run.game_id,
                        levels_completed=int(run.levels_completed or 0),
                    )
                    if _live:
                        print(
                            f'PORTFOLIO_LIVE_REALLOC game_id={run.game_id} '
                            f'levels={int(run.levels_completed or 0)} '
                            f'max_runtime_s_per_game={_live}',
                            flush=True,
                        )
                if run.game_id in emitted or str(run.state) == "playing":
                    continue
                row = _gate1_persist_run(run, terminal=True)
                _gate1_write_progress_index()
                try:
                    bm._save_json()
                except Exception as exc:
                    print(f"GATE1_INCREMENTAL_BENCHMARK_SAVE_ERROR {exc!r}", flush=True)
                emitted.add(run.game_id)
                print(
                    "GATE1_GAME_COMPLETE "
                    + json.dumps(
                        {
                            "game_id": row["game_id"],
                            "state": row["state"],
                            "levels_completed": row["levels_completed"],
                            "actions_taken": row["actions_taken"],
                            "elapsed_seconds": row["persisted_elapsed_seconds"],
                        },
                        sort_keys=True,
                    ),
                    flush=True,
                )
            await _gate1_asyncio.sleep(0.25)
    finally:
        for run in list(bm.game_runs):
            if run.game_id not in emitted and str(run.state) != "playing":
                row = _gate1_persist_run(run, terminal=True)
                print(
                    "GATE1_GAME_COMPLETE "
                    + json.dumps(
                        {
                            "game_id": row["game_id"],
                            "state": row["state"],
                            "levels_completed": row["levels_completed"],
                            "actions_taken": row["actions_taken"],
                            "elapsed_seconds": row["persisted_elapsed_seconds"],
                        },
                        sort_keys=True,
                    ),
                    flush=True,
                )
        _gate1_write_progress_index()


def _gate1_proc_start_ticks(pid):
    try:
        raw = Path(f"/proc/{pid}/stat").read_text()
        close = raw.rfind(")")
        fields = raw[close + 2 :].split()
        return int(fields[19])
    except Exception:
        return None


def _gate1_gpu_rows():
    try:
        completed = subprocess.run(
            [
                "nvidia-smi",
                "--query-compute-apps=pid,process_name,used_memory,gpu_uuid",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            check=False,
            timeout=3.0,
        )
    except Exception as exc:
        return [{"query_error": repr(exc)}]
    if completed.returncode != 0:
        return [{"query_error": completed.stderr[-2000:], "returncode": completed.returncode}]
    result = []
    for line in completed.stdout.splitlines():
        parts = [part.strip() for part in line.split(",", 3)]
        if len(parts) == 4 and parts[0].isdigit():
            result.append(
                {
                    "pid": int(parts[0]),
                    "process_name": parts[1],
                    "used_memory_mib": parts[2],
                    "gpu_uuid": parts[3],
                    "start_ticks": _gate1_proc_start_ticks(int(parts[0])),
                }
            )
    return result


def _gate1_kill_exact(rows):
    killed = []
    for row in rows:
        pid = int(row.get("pid", -1))
        saved_ticks = row.get("start_ticks")
        current_ticks = _gate1_proc_start_ticks(pid)
        if pid <= 1 or saved_ticks is None or current_ticks != int(saved_ticks):
            continue
        try:
            _gate1_os.kill(pid, _gate1_signal.SIGKILL)
            killed.append({"pid": pid, "start_ticks": current_ticks})
        except ProcessLookupError:
            pass
    return killed


def _gate1_teardown_once(label):
    attempt = {"label": label, "commands": []}
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command ({label}): {command}", flush=True)
        try:
            completed = subprocess.run(
                command,
                shell=True,
                check=False,
                cwd=WORKING_DIR,
                env=_command_env(),
                timeout=90.0,
            )
            attempt["commands"].append(
                {"command": command, "returncode": int(completed.returncode)}
            )
        except Exception as exc:
            attempt["commands"].append({"command": command, "error": repr(exc)})
    result_path = WORKING_DIR / "vllm-server-teardown.json"
    if result_path.is_file():
        try:
            attempt["result"] = json.loads(result_path.read_text())
        except Exception as exc:
            attempt["result_error"] = repr(exc)
    else:
        attempt["result_error"] = "result artifact missing"
    _gate1_teardown_attempts.append(attempt)
    return attempt


def _gate1_teardown_with_recovery():
    first = _gate1_teardown_once("initial")
    first_result = first.get("result") or {}
    first_codes_ok = all(item.get("returncode") == 0 for item in first["commands"])
    if first_codes_ok and first_result.get("shutdown_ok") is True:
        return True, first_result, _gate1_gpu_rows()

    owned_rows = list(first_result.get("vllm_gpu_rows_after") or [])
    owned_identity = {
        int(row["pid"]): row.get("start_ticks")
        for row in owned_rows
        if "pid" in row
    }
    fallback_kills = _gate1_kill_exact(owned_rows)
    drain_started = time.monotonic()
    drain_deadline = drain_started + 45.0
    remaining = []
    while True:
        gpu_rows = _gate1_gpu_rows()
        remaining = [row for row in gpu_rows if row.get("pid") in owned_identity]
        if not remaining or time.monotonic() >= drain_deadline:
            break
        retry_rows = [
            {"pid": row["pid"], "start_ticks": owned_identity.get(row["pid"])}
            for row in remaining
        ]
        fallback_kills.extend(_gate1_kill_exact(retry_rows))
        time.sleep(0.5)
    print(
        "GATE1_GPU_DRAIN "
        + json.dumps(
            {
                "owned_pids": sorted(owned_identity),
                "fallback_kills": fallback_kills,
                "wait_seconds": time.monotonic() - drain_started,
                "remaining": remaining,
            },
            sort_keys=True,
        ),
        flush=True,
    )

    second = _gate1_teardown_once("post_gpu_drain")
    second_result = second.get("result") or {}
    second_codes_ok = all(item.get("returncode") == 0 for item in second["commands"])
    post_rows = _gate1_gpu_rows()
    post_owned = [row for row in post_rows if row.get("pid") in owned_identity]
    ok = second_codes_ok and second_result.get("shutdown_ok") is True and not post_owned
    return ok, second_result, post_rows


if GATE1_SMOKE_MODE:
    hard_end_epoch = NOTEBOOK_START_EPOCH + GATE1_SMOKE_NOTEBOOK_LIMIT_SECONDS
    soft_end_epoch = hard_end_epoch - 30.0
else:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    required_reserve = GATE1_SAFETY_MARGIN_SECONDS + GATE1_SOFT_STOP_GRACE_SECONDS
    if budget <= required_reserve:
        raise RuntimeError(f"Notebook budget is too small for the Gate 1 reserve: {budget}.")
    hard_end_epoch = NOTEBOOK_START_EPOCH + budget - GATE1_SAFETY_MARGIN_SECONDS
    soft_end_epoch = hard_end_epoch - GATE1_SOFT_STOP_GRACE_SECONDS
soft_end = datetime.fromtimestamp(soft_end_epoch)
print(
    f"GATE1_DEADLINES smoke={GATE1_SMOKE_MODE} soft_end_epoch={soft_end_epoch} "
    f"hard_end_epoch={hard_end_epoch}",
    flush=True,
)

if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))
import vllm_server_watchdog as vllm_watchdog

vllm_watchdog_setup = vllm_watchdog.load_setup(BUNDLE_DIR / "serving_setup.py")
vllm_watchdog.start_background(
    vllm_watchdog_setup,
    vllm_watchdog.WatchdogConfig(
        interval_seconds=15.0,
        request_timeout_seconds=5,
        failure_threshold=4,
        max_restart_attempts=2,
    ),
)

_gate1_progress_stop = _gate1_asyncio.Event()
_gate1_progress_task = _gate1_asyncio.create_task(
    _gate1_progress_loop(_gate1_progress_stop)
)
try:
    remaining_run_seconds = hard_end_epoch - time.time()
    if remaining_run_seconds <= 0:
        raise TimeoutError("No smoke/global runtime remained before benchmark start.")
    try:
        await _gate1_asyncio.wait_for(
            bm.run(
                soft_end_time=soft_end,
                runtime_environment=target,
                minimal_diagnostics=True,
            ),
            timeout=remaining_run_seconds,
        )
    except _gate1_asyncio.TimeoutError:
        _gate1_hard_guard_triggered = True
        raise TimeoutError(
            f"Hard runtime guard triggered at {time.time()} with end={hard_end_epoch}."
        )
except Exception as exc:
    _gate1_benchmark_error = repr(exc)
    print(f"GATE1_BENCHMARK_EXCEPTION {_gate1_benchmark_error}", flush=True)
finally:
    _gate1_progress_stop.set()
    await _gate1_asyncio.gather(_gate1_progress_task, return_exceptions=True)
    _gate1_flush_all_runs()

try:
    if _gate1_benchmark_error is None:
        if not TRUE_SUBMISSION:
            import pandas as pd

            pd.DataFrame(
                [["1_0", "1", True, 1]],
                columns=["row_id", "game_id", "end_of_game", "score"],
            ).to_parquet(WORKING_DIR / "submission.parquet", index=False)

            public_runs = list(bm.game_runs)
            public_run_ids = [run.game_id for run in public_runs]
            if len(public_runs) != len(_gate1_expected_ids) or public_run_ids != _gate1_expected_ids:
                raise RuntimeError(
                    f"Public run coverage changed: count={len(public_runs)} ids={public_run_ids}."
                )
            unfinished = [
                (run.game_id, run.state, run.final_score)
                for run in public_runs
                if str(run.state) not in {"won", "gave_up", "cancelled"}
                or run.final_score is None
            ]
            if unfinished:
                raise RuntimeError(f"Public runs did not finalize cleanly: {unfinished}.")
            total_actions = sum(len(run.history) for run in public_runs)
            if total_actions <= 0:
                raise RuntimeError("Public runs produced no actions.")

            from inference.tools.eval import evaluate_runs, save_score_file

            score_summary = evaluate_runs([WORKING_DIR])
            score_path = save_score_file(
                score_summary,
                run_dirs=[WORKING_DIR],
                output_path=WORKING_DIR / "score.json",
            )
            if Path(score_path) != WORKING_DIR / "score.json" or not Path(score_path).is_file():
                raise RuntimeError(f"Frozen scorer did not write score.json: {score_path}.")
            print(
                f"PUBLIC_AUDIT smoke={GATE1_SMOKE_MODE} runs={len(public_runs)} "
                f"actions={total_actions} score_path={score_path}",
                flush=True,
            )
        _gate1_benchmark_ok = True
except Exception as exc:
    _gate1_benchmark_error = repr(exc)
    _gate1_benchmark_ok = False
    print(f"GATE1_BENCHMARK_VALIDATION_EXCEPTION {_gate1_benchmark_error}", flush=True)
finally:
    _gate1_flush_all_runs()

try:
    vllm_watchdog.stop_background(timeout_seconds=15.0)
except Exception as exc:
    print(f"GATE1_WATCHDOG_STOP_ERROR {exc!r}", flush=True)

try:
    _gate1_teardown_ok, _gate1_teardown_result, _gate1_post_gpu_rows = (
        _gate1_teardown_with_recovery()
    )
    if not _gate1_teardown_ok:
        _gate1_teardown_error = "bounded terminal gate did not pass after recovery"
except Exception as exc:
    _gate1_teardown_error = repr(exc)
    _gate1_teardown_ok = False
    _gate1_post_gpu_rows = _gate1_gpu_rows()

_gate1_lifecycle = {
    "smoke_mode": GATE1_SMOKE_MODE,
    "benchmark_ok": _gate1_benchmark_ok,
    "benchmark_error": _gate1_benchmark_error,
    "teardown_ok": _gate1_teardown_ok,
    "teardown_error": _gate1_teardown_error,
    "hard_guard_triggered": _gate1_hard_guard_triggered,
    "teardown_attempts": _gate1_teardown_attempts,
    "post_teardown_gpu_rows": _gate1_post_gpu_rows,
    "elapsed_seconds": time.monotonic() - _GATE1_PROGRESS_START,
}
# Portfolio coverage artifacts. Session-end notify already ran per game (live).
# Only backfill notify for games missed by the session hook (should be rare).
_portfolio_rows = []
_notified = globals().get('_PORTFOLIO_SESSION_NOTIFIED') or set()
for _run in list(getattr(bm, "game_runs", []) or []):
    _row = _gate1_run_snapshot(_run)
    _gid = _row['game_id']
    if _gid in _notified:
        _game = next((g for g in _PORTFOLIO_PLAN.games if g.game_id == _gid), None)
        _reason = (
            _game.abandon_reason
            if _game is not None
            else str((_GATE1_SESSION_METRICS.get(_gid) or {}).get('abandon_reason') or '')
        )
        _row['abandon_reason'] = _reason
        print(
            'PORTFOLIO_NOTIFY_SKIP_ALREADY_SESSION '
            + json.dumps({'game_id': _gid, 'abandon_reason': _reason}, sort_keys=True),
            flush=True,
        )
    else:
        _reason = notify_level_and_spend(
            _PORTFOLIO_PLAN,
            game_id=_gid,
            wall_seconds=float(_row.get('active_wall_seconds') or 0.0),
            levels_completed=int(_row.get('levels_completed') or 0),
            actions=int(_row.get('actions_taken') or 0),
        )
        _row['abandon_reason'] = _reason
        _notified.add(_gid)
        print(
            'PORTFOLIO_NOTIFY_BACKFILL '
            + json.dumps(
                {
                    'game_id': _gid,
                    'actions': _row.get('actions_taken'),
                    'levels': _row.get('levels_completed'),
                    'wall': _row.get('active_wall_seconds'),
                    'abandon_reason': _reason,
                },
                sort_keys=True,
            ),
            flush=True,
        )
    _portfolio_rows.append(_row)
_trace_path = os.environ.get('ARC3_TRACE') or str(WORKING_DIR / 'coverage.jsonl')
_trace_summary = record_duck_rows(
    _portfolio_rows,
    trace_path=_trace_path,
    scored_environments=int(os.environ.get('ARC3_SCORED_ENVIRONMENTS', '25')),
)
(WORKING_DIR / 'coverage.summary.json').write_text(
    json.dumps(_trace_summary, indent=2, sort_keys=True) + '\n',
    encoding='utf-8',
)
(WORKING_DIR / 'portfolio_plan_final.json').write_text(
    json.dumps(
        {
            'games': [
                {
                    'game_id': g.game_id,
                    'allocated_seconds': g.allocated_seconds,
                    'spent_seconds': g.spent_seconds,
                    'levels_completed': g.levels_completed,
                    'abandon_reason': g.abandon_reason,
                    'phase': g.phase.value if hasattr(g.phase, 'value') else str(g.phase),
                }
                for g in _PORTFOLIO_PLAN.games
            ],
            'reserve_seconds': _PORTFOLIO_PLAN.reserve_seconds,
        },
        indent=2,
        sort_keys=True,
    )
    + '\n',
    encoding='utf-8',
)
print('PORTFOLIO_TRACE ' + json.dumps(_trace_summary, sort_keys=True), flush=True)
print('PORTFOLIO_MINIWAVE_ARTIFACTS coverage.jsonl coverage.summary.json portfolio_plan.json portfolio_plan_final.json', flush=True)

_gate1_atomic_json(WORKING_DIR / "gate1_lifecycle.json", _gate1_lifecycle)
print(
    "GATE1_LIFECYCLE "
    + json.dumps(
        {
            "benchmark": "ok" if _gate1_benchmark_ok else "failed",
            "benchmark_error": _gate1_benchmark_error,
            "teardown": "ok" if _gate1_teardown_ok else "failed",
            "teardown_error": _gate1_teardown_error,
            "hard_guard_triggered": _gate1_hard_guard_triggered,
            "post_teardown_gpu_rows": _gate1_post_gpu_rows,
            "elapsed_seconds": _gate1_lifecycle["elapsed_seconds"],
        },
        sort_keys=True,
    ),
    flush=True,
)


In [ ]:
# Gate 1 terminal audit. This cell reports independent benchmark and teardown facts.
import json as _g1_json
import pandas as _g1_pd

_g1_lifecycle_path = WORKING_DIR / "gate1_lifecycle.json"
_g1_progress_path = WORKING_DIR / "gate1_progress.json"
_g1_lifecycle = (
    _g1_json.loads(_g1_lifecycle_path.read_text())
    if _g1_lifecycle_path.is_file()
    else {"benchmark_ok": False, "teardown_ok": False, "error": "lifecycle artifact missing"}
)
_g1_rows = (
    _g1_json.loads(_g1_progress_path.read_text())
    if _g1_progress_path.is_file()
    else []
)

# The bundled terminal gate captures metrics before shutdown, then requires its owned GPU
# rows to be gone. If only the NVIDIA table was still draining, the lifecycle wrapper kills
# the exact PID/start-time pair and waits. Re-evaluate the original gate against that
# post-drain table; do not rerun endpoint capture after the server port is intentionally closed.
if not _g1_lifecycle.get("teardown_ok"):
    _g1_attempts = list(_g1_lifecycle.get("teardown_attempts") or [])
    _g1_first_teardown = (
        dict((_g1_attempts[0].get("result") or {})) if _g1_attempts else {}
    )
    _g1_post_gpu = list(_g1_lifecycle.get("post_teardown_gpu_rows") or [])
    _g1_owned_pids = {
        int(row["pid"])
        for row in (_g1_first_teardown.get("vllm_gpu_rows_after") or [])
        if "pid" in row
    }
    _g1_post_owned = [
        row for row in _g1_post_gpu if row.get("pid") in _g1_owned_pids
    ]
    _g1_final_scan = _g1_first_teardown.get("process_scan_final_gate") or {}
    _g1_identity_ok = bool(_g1_first_teardown.get("identity_valid"))
    _g1_conflict = bool(
        _g1_final_scan.get("root_conflict")
        or _g1_final_scan.get("saved_conflicts")
    )
    _g1_cpu_survivors = bool(
        _g1_final_scan.get("authorized_records")
        or _g1_final_scan.get("suspect_records")
        or _g1_first_teardown.get("full_proc_marker_survivors")
        or _g1_first_teardown.get("cpu_only_vllm_ple_marker_survivors")
    )
    _g1_gpu_query_error = any("query_error" in row for row in _g1_post_gpu)
    _g1_recovered_gate = all(
        (
            _g1_identity_ok,
            not _g1_conflict,
            bool(_g1_first_teardown.get("port_closed")),
            not _g1_cpu_survivors,
            not _g1_gpu_query_error,
            not _g1_post_owned,
            bool(_g1_first_teardown.get("final_metrics_preserved")),
            bool(_g1_first_teardown.get("required_artifacts_preserved")),
        )
    )
    if _g1_recovered_gate:
        _g1_first_teardown["gpu_rows_after"] = _g1_post_gpu
        _g1_first_teardown["gpu_query_error_after"] = _g1_gpu_query_error
        _g1_first_teardown["vllm_gpu_rows_after"] = _g1_post_owned
        _g1_first_teardown["shutdown_ok"] = True
        _g1_first_teardown["terminal_gate_recovered_after_gpu_drain"] = True
        _gate1_atomic_json(
            WORKING_DIR / "vllm-server-teardown.json", _g1_first_teardown
        )
        _g1_lifecycle["teardown_ok"] = True
        _g1_lifecycle["teardown_error"] = None
        _g1_lifecycle["terminal_gate_recovered_after_gpu_drain"] = True
        _gate1_atomic_json(_g1_lifecycle_path, _g1_lifecycle)
        print(
            "GATE1_TEARDOWN_RECOVERED "
            + _g1_json.dumps(
                {
                    "owned_pids": sorted(_g1_owned_pids),
                    "post_owned": _g1_post_owned,
                    "post_gpu_rows": _g1_post_gpu,
                    "shutdown_ok": True,
                },
                sort_keys=True,
            ),
            flush=True,
        )

for _g1_row in _g1_rows:
    print("GATE1_GAME_TRACE " + _g1_json.dumps(_g1_row, sort_keys=True), flush=True)

_g1_summary = {
    "games_expected": len(_gate1_expected_ids),
    "games_persisted": len(_g1_rows),
    "games_terminal": sum(bool(row.get("terminal")) for row in _g1_rows),
    "games_with_level": sum(int(row.get("levels_completed", 0)) > 0 for row in _g1_rows),
    "levels_completed": sum(int(row.get("levels_completed", 0)) for row in _g1_rows),
    "total_actions": sum(int(row.get("actions_taken", 0)) for row in _g1_rows),
    "total_llm_calls": sum(int(row.get("llm_calls", 0)) for row in _g1_rows),
    "total_generated_tokens": sum(int(row.get("generated_tokens", 0)) for row in _g1_rows),
}
print("GATE1_COVERAGE " + _g1_json.dumps(_g1_summary, sort_keys=True), flush=True)

_g1_score_path = WORKING_DIR / "score.json"
_g1_score = None
if _g1_score_path.is_file():
    _g1_score_doc = _g1_json.loads(_g1_score_path.read_text())
    _g1_score = _g1_score_doc.get("score")
print("GATE1_SCORE " + _g1_json.dumps({"score": _g1_score}, sort_keys=True), flush=True)

_g1_submission_path = WORKING_DIR / "submission.parquet"
_g1_schema_ok = False
_g1_schema = {"exists": _g1_submission_path.is_file()}
if _g1_submission_path.is_file():
    try:
        _g1_submission = _g1_pd.read_parquet(_g1_submission_path)
        _g1_schema.update(
            {
                "columns": list(_g1_submission.columns),
                "dtypes": {
                    column: str(dtype)
                    for column, dtype in _g1_submission.dtypes.items()
                },
                "rows": len(_g1_submission),
            }
        )
        _g1_schema_ok = list(_g1_submission.columns) == [
            "row_id", "game_id", "end_of_game", "score"
        ]
    except Exception as exc:
        _g1_schema["error"] = repr(exc)
print("GATE1_SUBMISSION_SCHEMA " + _g1_json.dumps(_g1_schema, sort_keys=True), flush=True)

_g1_identity_path = WORKING_DIR / "vllm-server-identity.json"
_g1_tuning = {}
if _g1_identity_path.is_file():
    try:
        _g1_tuning = (
            _g1_json.loads(_g1_identity_path.read_text()).get("vllm_tuning") or {}
        )
    except Exception as exc:
        _g1_tuning = {"error": repr(exc)}
print("GATE1_VLLM_TUNING " + _g1_json.dumps(_g1_tuning, sort_keys=True), flush=True)

_g1_first_request = _GATE1_METRICS.get("first_request_started_epoch")
_g1_first_response = _GATE1_METRICS.get("first_response_epoch")
_g1_cold_seconds = (
    _g1_first_response - NOTEBOOK_START_EPOCH
    if _g1_first_response is not None
    else None
)
_g1_cold = {
    "notebook_start_epoch": NOTEBOOK_START_EPOCH,
    "first_request_started_epoch": _g1_first_request,
    "first_response_epoch": _g1_first_response,
    "notebook_to_first_response_seconds": _g1_cold_seconds,
    "notebook_to_first_response_fraction_of_32400": (
        _g1_cold_seconds / 32400.0 if _g1_cold_seconds is not None else None
    ),
}
print("GATE1_COLD_START " + _g1_json.dumps(_g1_cold, sort_keys=True), flush=True)

_g1_fit_ok = False
_g1_fit = {"cold_start_seconds": _g1_cold_seconds}
if _g1_cold_seconds is not None:
    _g1_projected_seconds = _g1_cold_seconds + 4.0 * GATE1_WAVE_CAP_SECONDS
    _g1_projected_margin = float(target.max_runtime_s) - _g1_projected_seconds
    _g1_fit.update(
        {
            "wave_cap_seconds": GATE1_WAVE_CAP_SECONDS,
            "waves": 4,
            "projected_seconds": _g1_projected_seconds,
            "projected_margin_seconds": _g1_projected_margin,
            "required_margin_seconds": GATE1_SAFETY_MARGIN_SECONDS,
        }
    )
    _g1_fit_ok = _g1_projected_margin >= GATE1_SAFETY_MARGIN_SECONDS
print("GATE1_PROJECTED_FIT " + _g1_json.dumps(_g1_fit, sort_keys=True), flush=True)

_g1_nvidia = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
    check=False,
    timeout=10.0,
)
print("GATE1_POST_NVIDIA_SMI_BEGIN", flush=True)
print(_g1_nvidia.stdout, flush=True)
print(_g1_nvidia.stderr, flush=True)
print("GATE1_POST_NVIDIA_SMI_END", flush=True)

_g1_measurement_ok = bool(
    _g1_lifecycle.get("benchmark_ok")
    and len(_g1_rows) == len(_gate1_expected_ids)
    and all(bool(row.get("terminal")) for row in _g1_rows)
    and _g1_schema_ok
    and _g1_score_path.is_file()
)
_g1_teardown_ok = bool(_g1_lifecycle.get("teardown_ok"))
_g1_clean = bool(
    _g1_measurement_ok
    and _g1_teardown_ok
    and not _g1_lifecycle.get("hard_guard_triggered")
)

print(
    "GATE1_FINAL_STATUS "
    + _g1_json.dumps(
        {
            "benchmark": "ok" if _g1_measurement_ok else "failed",
            "teardown": "ok" if _g1_teardown_ok else "failed",
            "teardown_error": _g1_lifecycle.get("teardown_error"),
            "logical_exit_code": 0 if _g1_clean else 1,
            "post_teardown_gpu_rows": _g1_lifecycle.get("post_teardown_gpu_rows"),
            "smoke_mode": GATE1_SMOKE_MODE,
        },
        sort_keys=True,
    ),
    flush=True,
)

if GATE1_SMOKE_MODE:
    _portfolio_trace_ok = Path(os.environ.get('ARC3_TRACE') or (WORKING_DIR / 'coverage.jsonl')).is_file()
    _portfolio_actions = sum(int(r.get('actions_taken') or 0) for r in _g1_rows)
    _portfolio_ok = (
        _g1_clean
        and len(_g1_rows) == GATE1_SMOKE_GAME_COUNT
        and _portfolio_trace_ok
        and _portfolio_actions > 0
    )
    print(
        'PORTFOLIO_SMOKE_STATUS '
        + json.dumps(
            {
                'clean': _g1_clean,
                'rows': len(_g1_rows),
                'expected': GATE1_SMOKE_GAME_COUNT,
                'actions': _portfolio_actions,
                'trace_ok': _portfolio_trace_ok,
                'pass': _portfolio_ok,
            },
            sort_keys=True,
        ),
        flush=True,
    )
    if _portfolio_ok:
        print("GATE1_SMOKE_OK", flush=True)
        print("PORTFOLIO_MINIWAVE_OK", flush=True)
    else:
        print("GATE1_SMOKE_FAILED", flush=True)
        print("PORTFOLIO_MINIWAVE_FAILED", flush=True)
else:
    if _g1_clean and _g1_fit_ok:
        print("GATE1_VALIDATION_OK", flush=True)
    else:
        print("GATE1_VALIDATION_BLOCKED", flush=True)
